# Etapa 1: Análise exploratória

Dados de treino da competição **Recod.ai/LUC: Scientific Image Forgery Detection**. Cada imagem é
**autêntica** ou **forjada** por *copy-move* (uma região da figura foi copiada e colada em outro lugar da
mesma figura). As forjadas vêm com uma ou mais máscaras binárias.

**Termos usados no notebook.** Cada máscara é uma **instância**: um *par de cópias*, que marca ao mesmo
tempo a **origem** (o trecho copiado) e o **destino** (onde ele foi colado). Por isso uma máscara costuma ter
duas manchas separadas, que chamo de **pedaços** (componentes conexos). Uma imagem com duas máscaras tem
duas cópias independentes. É a instância, e não o pedaço, que a métrica casa entre predição e verdade.

O notebook responde, nesta ordem:
1. Quantas imagens há de cada classe, e como os `case_id` se repetem entre as classes.
2. Tamanhos e proporções das imagens.
3. Número de instâncias por imagem forjada e de pedaços por instância.
4. Área relativa das instâncias e dos pedaços (fração da imagem coberta).
5. Exemplos de forjadas com as máscaras sobrepostas.
6. O que torna o problema difícil, com uma medida para cada fator: figuras compostas, cópias pequenas,
   cópias com rotação/escala/espelhamento, texturas repetitivas que não são fraude.

As figuras são salvas em `results/eda/` para o relatório.

In [ ]:
import os, sys
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.insert(0, os.path.abspath(".."))          # para importar pipeline/ da pasta do trabalho
from pipeline.common import load_config
import eda

cfg = load_config()
DATA = os.environ.get("T3_DATA", cfg["data_dir"])  # T3_DATA permite apontar para outra pasta
OUT = os.path.join(cfg["results_dir"], "eda")
os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(cfg["seed"])

def save(fig, name):
    fig.savefig(os.path.join(OUT, name + ".png"), dpi=150, bbox_inches="tight")

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
CLASS_COLOR = {"authentic": "#4C72B0", "forged": "#DD5143"}

df = eda.list_images(DATA)
assert len(df), f"nenhuma imagem em {DATA}: rode ./download_data.sh"
print(f"{len(df)} imagens em {DATA}")
df.head()

## 1. Contagens

**Por que olhar o `case_id`?** O nome do arquivo é o identificador do caso. Se a mesma figura aparece
como autêntica *e* como forjada (a forjada sendo a versão adulterada da autêntica), as duas precisam
ficar na mesma partição do split. Senão o modelo vê a versão original no treino e é testado na
adulterada, o que é vazamento de informação.

In [ ]:
counts = pd.crosstab(df["origem"], df["classe"], margins=True, margins_name="total")
display(counts)

by_case = df.groupby("case_id")["classe"].agg(lambda s: "+".join(sorted(set(s))))
display(by_case.value_counts().rename("n_case_ids").to_frame())

fig, ax = plt.subplots(figsize=(4, 3))
vc = df["classe"].value_counts()
ax.bar(vc.index, vc.values, color=[CLASS_COLOR[c] for c in vc.index])
for i, v in enumerate(vc.values):
    ax.text(i, v, f"{v}\n({v / len(df):.0%})", ha="center", va="bottom")
ax.set_ylabel("imagens"); ax.set_title("Imagens por classe")
save(fig, "1_classes")

## 2. Estatísticas por imagem

Para cada imagem: altura, largura, megapixels, proporção (largura/altura), se é em tons de cinza,
número estimado de painéis e, nas forjadas, número de instâncias, pedaços por instância, área de cada
instância e de cada pedaço, área da união e o lado do menor pedaço.
O cálculo lê todas as imagens e máscaras uma vez e fica guardado em `results/eda/stats.pkl`.

In [ ]:
cache = os.path.join(OUT, "stats.pkl")
if os.path.exists(cache):
    stats = pd.read_pickle(cache)
else:
    stats = pd.DataFrame([eda.image_stats(r) for _, r in tqdm(df.iterrows(), total=len(df))])
    stats = pd.concat([df.reset_index(drop=True), stats], axis=1)
    stats.to_pickle(cache)
forged = stats[stats["classe"] == "forged"]
authentic = stats[stats["classe"] == "authentic"]
stats.groupby("classe")[["altura", "largura", "megapixels", "proporcao"]].describe().T.round(2)

### 2.1 Tamanhos e proporções

Os tamanhos variam muito, então os eixos estão em escala logarítmica. Isso pesa em duas decisões:
o tamanho de entrada da rede (redimensionar uma imagem grande apaga regiões pequenas) e a memória
da GPU (6 GB). Também é preciso checar se autênticas e forjadas têm a mesma distribuição de tamanho:
se não tiverem, o modelo pode aprender a classe pelo tamanho, o que é um atalho que não generaliza.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.3))
bins = np.logspace(np.log10(stats["megapixels"].min()), np.log10(stats["megapixels"].max()), 40)
for c, g in stats.groupby("classe"):
    axs[0].hist(g["megapixels"], bins=bins, alpha=0.6, label=c, color=CLASS_COLOR[c])
    axs[1].scatter(g["largura"], g["altura"], s=4, alpha=0.4, color=CLASS_COLOR[c], label=c)
    axs[2].hist(np.log2(g["proporcao"]), bins=40, alpha=0.6, color=CLASS_COLOR[c], label=c)
axs[0].set(xscale="log", xlabel="megapixels", ylabel="imagens", title="Tamanho")
axs[1].set(xscale="log", yscale="log", xlabel="largura (px)", ylabel="altura (px)", title="Largura x altura")
axs[2].set(xlabel="log2(largura / altura)", title="Proporção (0 = quadrada)")
axs[0].legend()
save(fig, "2_tamanhos")

print("lado maior: mediana {:.0f} px, 5% {:.0f} px, 95% {:.0f} px".format(
    *np.percentile(stats[["altura", "largura"]].max(axis=1), [50, 5, 95])))
print(f"imagens em tons de cinza: {stats.groupby('classe')['cinza'].mean().round(3).to_dict()}")

## 3. Instâncias por imagem e pedaços por instância

A métrica casa cada máscara prevista com uma verdadeira e divide por max(previstas, verdadeiras):
prever menos ou mais instâncias do que existem custa caro. Por isso importa saber quantas instâncias
são típicas e como cada uma é formada.

O gráfico da direita mostra que quase toda máscara tem **dois pedaços**: a origem e o destino da mesma
cópia ficam na mesma máscara. Três ou mais pedaços indicam, por exemplo, uma origem colada em vários
destinos ou uma cópia que cruza a borda de um painel. Consequência direta para o pós-processamento:
separar a predição em componentes conexos e devolver cada um como uma instância dobraria o número de
máscaras e cortaria o escore pela metade. Os pedaços precisam ser **agrupados em pares**.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
vc = forged["n_instancias"].value_counts().sort_index()
axs[0].bar(vc.index.astype(str), vc.values, color=CLASS_COLOR["forged"])
for i, v in enumerate(vc.values):
    axs[0].text(i, v, f"{v / len(forged):.0%}", ha="center", va="bottom")
axs[0].set(xlabel="instâncias (máscaras) por imagem", ylabel="imagens forjadas", title="Instâncias por imagem")

comps = Counter(min(c, 5) for cs in forged["pedacos_por_instancia"] for c in cs)
axs[1].bar([str(k) if k < 5 else "5+" for k in sorted(comps)], [comps[k] for k in sorted(comps)], color="gray")
axs[1].set(xlabel="pedaços (componentes conexos) por máscara", ylabel="máscaras", title="Pedaços por instância")
save(fig, "3_instancias")

## 4. Área relativa das instâncias e dos pedaços

Área relativa = fração dos pixels da imagem coberta. A área de uma instância soma a origem e o destino,
então cada pedaço tem, em geral, cerca de metade dela. Cópias muito pequenas são difíceis de achar e
quase não pesam numa perda de segmentação comum (como entropia cruzada por pixel), porque o fundo
domina. O painel da direita mostra o **lado menor da caixa** do menor pedaço de cada imagem, em pixels:
é o que sobra para a rede enxergar depois de redimensionar.

In [ ]:
areas = np.concatenate(forged["areas_instancias"].to_numpy())
areas_pedacos = np.concatenate(forged["areas_pedacos"].to_numpy())
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
bins = np.logspace(np.log10(max(areas_pedacos.min(), 1e-5)), 0, 40)
axs[0].hist(areas_pedacos, bins=bins, alpha=0.5, label="cada pedaço", color="lightgray")
axs[0].hist(areas, bins=bins, alpha=0.7, label="cada instância", color="gray")
axs[0].hist(forged["area_uniao"], bins=bins, alpha=0.6, label="união na imagem", color=CLASS_COLOR["forged"])
axs[0].set(xscale="log", xlabel="fração da imagem", ylabel="contagem", title="Área relativa")
axs[0].legend()
axs[1].hist(forged["lado_min_pedaco_px"].dropna(), bins=np.logspace(0, 4, 40), color=CLASS_COLOR["forged"])
axs[1].set(xscale="log", xlabel="lado menor da caixa do menor pedaço (px, resolução original)",
           title="Tamanho do menor pedaço")
save(fig, "4_areas")

for p in [0.001, 0.01, 0.05]:
    print(f"pedaços com menos de {p:.1%} da imagem: {np.mean(areas_pedacos < p):.1%}")
print("área por instância: mediana {:.2%}, quartis {:.2%} a {:.2%}".format(*np.percentile(areas, [50, 25, 75])))
print("área por pedaço:    mediana {:.2%}, quartis {:.2%} a {:.2%}".format(*np.percentile(areas_pedacos, [50, 25, 75])))

## 5. Exemplos de forjadas com as máscaras

Cada cor é uma instância (uma máscara): os dois pedaços da mesma cor são a origem e o destino da cópia. Quando o mesmo `case_id` existe como autêntica,
ela aparece ao lado, o que mostra o que foi adulterado.

In [ ]:
def show_case(row, axs):
    img = eda.load_image(row["path"])
    masks = eda.load_masks(row["mask_path"], img.shape[:2])
    pair = stats[(stats["case_id"] == row["case_id"]) & (stats["classe"] == "authentic")]
    if len(pair):
        axs[0].imshow(eda.load_image(pair.iloc[0]["path"]), cmap="gray"); axs[0].set_title("autêntica")
    axs[1].imshow(img); axs[1].set_title(f"forjada {row['case_id']}")
    axs[2].imshow(eda.overlay(img, masks))
    axs[2].set_title(f"{len(masks)} instância(s), união {row['area_uniao']:.1%}")
    for a in axs: a.axis("off")

n = min(cfg.get("eda_examples", 8), len(forged))
sample = forged.sample(n, random_state=cfg["seed"])
fig, axs = plt.subplots(n, 3, figsize=(10, 3.2 * n), squeeze=False)
for (_, row), ax in zip(sample.iterrows(), axs):
    show_case(row, ax)
fig.tight_layout(); save(fig, "5_exemplos")

## 6. O que torna o problema difícil

Para cada fator do enunciado, uma medida tirada dos dados.

### 6.1 Figuras compostas

Figuras científicas costumam juntar vários painéis (A, B, C...) separados por faixas de fundo. Painéis
parecidos entre si (o mesmo tipo de célula, o mesmo blot) são comuns e legítimos, e confundem um
detector de duplicação. A contagem abaixo é uma estimativa: linhas e colunas quase inteiras da cor do
fundo são tratadas como separadores.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
for i, (c, g) in enumerate(stats.groupby("classe")):
    vc = g["n_paineis"].clip(upper=10).value_counts(normalize=True).sort_index()
    ax.bar(vc.index + (i - 0.5) * 0.4, vc.values, width=0.4, color=CLASS_COLOR[c], label=c)
ax.set(xlabel="painéis estimados (10 = 10 ou mais)", ylabel="fração das imagens", title="Figuras compostas")
ax.legend(); save(fig, "6_1_paineis")
print("fração com mais de um painel:", (stats["n_paineis"] > 1).groupby(stats["classe"]).mean().round(3).to_dict())

# a forjada com mais paineis, para ver como a estimativa se comporta
row = forged.sort_values("n_paineis").iloc[-1]
fig, axs = plt.subplots(1, 3, figsize=(10, 3.2)); show_case(row, axs)

### 6.2 Cópias pequenas versus resolução de entrada

Redes de segmentação recebem a imagem redimensionada para um tamanho fixo. A tabela mostra em que
fração das forjadas o menor pedaço fica com o lado menor abaixo de 8 e de 16 pixels em cada resolução de entrada
(8 px é aproximadamente uma célula do mapa de atributos de um ViT com *patch* de 14 ou de uma CNN com
passo 8). Isso orienta a escolha entre redimensionar e processar em blocos (*tiles*).

In [ ]:
rows = []
for side in [256, 384, 512, 768, 1024]:
    scale = side / forged[["altura", "largura"]].max(axis=1)
    small = forged["lado_min_pedaco_px"] * scale
    rows.append({"lado da entrada": side, "< 8 px": f"{np.mean(small < 8):.1%}", "< 16 px": f"{np.mean(small < 16):.1%}"})
pd.DataFrame(rows).set_index("lado da entrada")

### 6.3 Cópias com rotação, escala ou espelhamento

Para cada instância com pelo menos dois pedaços, recorto os dois maiores (origem e destino, sem saber
qual é qual), aplico ao segundo cada uma das 8 orientações
possíveis (identidade, espelhos, rotações de 90°, transpostas), redimensiono para o tamanho da
primeira e meço a correlação. A orientação com maior correlação é a estimativa de como a cópia foi
colada. A razão de escala vem da raiz da razão entre as áreas.

Correlação baixa mesmo na melhor orientação indica que a cópia foi alterada depois de colada
(contraste, ruído, compressão) ou que os dois pedaços não correspondem pixel a pixel.

In [ ]:
tr = []
for _, row in tqdm(forged.iterrows(), total=len(forged)):
    img = eda.load_image(row["path"])
    for mask in eda.load_masks(row["mask_path"], img.shape[:2]):
        r = eda.copy_transform(img, mask)
        if r: tr.append({"case_id": row["case_id"], **r})
tr = pd.DataFrame(tr)
print(f"{len(tr)} instâncias com dois pedaços analisáveis, de {forged['n_instancias'].sum()} instâncias")

confident = tr[tr["corr_melhor"] > 0.8]       # so as copias claramente reconhecidas
fig, axs = plt.subplots(1, 3, figsize=(14, 3.3))
vc = confident["melhor_orientacao"].value_counts()
axs[0].barh(vc.index[::-1], vc.values[::-1], color=CLASS_COLOR["forged"])
axs[0].set(xlabel="pares (correlação > 0,8)", title="Orientação da cópia")
axs[1].hist(np.log2(tr["escala"]).clip(-2, 2), bins=np.linspace(-2, 2, 41), color="gray")
axs[1].set(xlabel="log2(escala entre origem e destino), cortado em ±2", title="Escala (0 = mesmo tamanho)")
axs[2].hist(tr["corr_melhor"].clip(0, 1), bins=np.linspace(0, 1, 41), color="gray")
axs[2].set(xlabel="correlação na melhor orientação", title="Fidelidade da cópia")
fig.tight_layout(); save(fig, "6_3_transformacoes")
print(f"pares com correlação > 0,8: {len(confident) / max(len(tr), 1):.1%}")
print(f"entre eles, sem rotação nem espelho: {(confident['melhor_orientacao'] == 'identidade').mean():.1%}")
print(f"escala diferente de 1 (mais de 10%): {np.mean(np.abs(np.log(tr['escala'])) > np.log(1.1)):.1%}")

### 6.4 O que a máscara marca: comparação com a autêntica do mesmo `case_id`

Quando existe a versão autêntica da mesma figura, a diferença pixel a pixel mostra onde a imagem foi
alterada. Se só o trecho colado muda, a fração alterada de cada **pedaço** diz se ele é a **origem**
(não muda) ou o **destino** (muda) da cópia. Se cada instância tem um pedaço intacto e um alterado, isso
confirma que a máscara marca as duas pontas da cópia. Isso importa porque o modelo precisa segmentar as
duas, e a origem é idêntica à imagem autêntica: ela só pode ser encontrada pela semelhança com o destino.

In [ ]:
auth_path = authentic.set_index("case_id")["path"]
diffs = []
for _, row in tqdm(forged[forged["case_id"].isin(auth_path.index)].iterrows()):
    img_shape = (row["altura"], row["largura"])
    d = eda.diff_vs_authentic(row["path"], auth_path[row["case_id"]], eda.load_masks(row["mask_path"], img_shape))
    if d: diffs.append(d)
diffs = pd.DataFrame(diffs)
if len(diffs):
    per_piece = np.concatenate(diffs["por_pedaco_alterado"].to_numpy())
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(per_piece, bins=20, color=CLASS_COLOR["forged"])
    ax.set(xlabel="fração do pedaço alterada em relação à autêntica", ylabel="pedaços", title="Origem (0) ou destino (1)?")
    save(fig, "6_4_origem_destino")
    print(f"{len(diffs)} pares forjada/autêntica com o mesmo tamanho")
    print(f"pedaços quase intactos (<10% alterado, provável origem): {np.mean(per_piece < 0.1):.1%}")
    print(f"pedaços quase todo alterados (>90%, provável destino): {np.mean(per_piece > 0.9):.1%}")
    print(f"pixels alterados que caem fora das máscaras: mediana {diffs['frac_alterado_fora_mascara'].median():.1%}")
else:
    print("nenhum case_id com autêntica e forjada do mesmo tamanho")

### 6.5 Texturas repetitivas que não são fraude

Um detector clássico procura pontos-chave (SIFT) que se repetem dentro da própria imagem. Se as
autênticas também têm muitas repetições (células parecidas, padrões regulares de blots, texto e
eixos de gráfico), esse sinal sozinho gera falsos positivos, que a métrica pune com escore zero.

Para uma amostra de cada classe, conto os pares de pontos SIFT que casam com outro ponto da mesma
imagem (teste da razão de Lowe com o 2º e o 3º vizinho, e distância mínima de 20 px entre as pontas).
Nas forjadas, também meço a fração dos pares com as duas pontas dentro das máscaras.

In [ ]:
n_sift = min(300, len(authentic), len(forged))
sift = []
for _, row in tqdm(pd.concat([authentic.sample(n_sift, random_state=0), forged.sample(n_sift, random_state=0)]).iterrows(),
                   total=2 * n_sift):
    img = eda.load_image(row["path"])
    masks = eda.load_masks(row["mask_path"], img.shape[:2]) if row["classe"] == "forged" else None
    n_kp, n_pairs, inside = eda.sift_self_matches(img, masks)
    sift.append({"case_id": row["case_id"], "classe": row["classe"], "path": row["path"],
                 "n_pontos": n_kp, "n_pares": n_pairs, "frac_dentro_mascara": inside})
sift = pd.DataFrame(sift)

fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
bins = np.r_[0, np.logspace(0, np.log10(sift["n_pares"].max() + 1), 30)]
for c, g in sift.groupby("classe"):
    axs[0].hist(g["n_pares"] + 0.5, bins=bins + 0.5, alpha=0.6, color=CLASS_COLOR[c], label=c)
axs[0].set(xscale="log", xlabel="pares SIFT repetidos na própria imagem (+0,5)", ylabel="imagens", title="Autocorrespondência")
axs[0].legend()
axs[1].hist(sift["frac_dentro_mascara"].dropna(), bins=20, color=CLASS_COLOR["forged"])
axs[1].set(xlabel="fração dos pares dentro das máscaras", title="Forjadas: os pares caem na fraude?")
save(fig, "6_5_sift")

summary = sift.groupby("classe")["n_pares"].describe(percentiles=[0.5, 0.9])
summary["sem pares"] = sift.groupby("classe")["n_pares"].apply(lambda s: (s == 0).mean())
display(summary.round(2))

In [ ]:
# autentica com mais repeticoes: um exemplo de textura repetitiva legitima
row = sift[sift["classe"] == "authentic"].sort_values("n_pares").iloc[-1]
fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(eda.load_image(row["path"])); ax.axis("off")
ax.set_title(f"autêntica {row['case_id']}: {row['n_pares']} pares SIFT repetidos")
save(fig, "6_5_autentica_repetitiva")

## 7. Resumo em números

A célula abaixo junta os números usados na discussão.

In [ ]:
two_classes = by_case.str.contains(r"\+").sum()
print(f"- {len(df)} imagens: {len(authentic)} autênticas, {len(forged)} forjadas ({len(forged) / len(df):.0%})")
print(f"- {two_classes} case_id aparecem nas duas classes -> split agrupado por case_id é obrigatório")
print(f"- megapixels: mediana {stats['megapixels'].median():.2f}, de {stats['megapixels'].min():.2f} a {stats['megapixels'].max():.1f}")
print(f"- instâncias por forjada: {forged['n_instancias'].value_counts(normalize=True).sort_index().round(2).to_dict()}")
pedacos = pd.Series(np.concatenate(forged['pedacos_por_instancia'].to_numpy()))
print(f"- pedaços por instância: {pedacos.clip(upper=4).value_counts(normalize=True).sort_index().round(2).to_dict()} (4 = 4 ou mais)")
print(f"- área por pedaço: mediana {np.median(areas_pedacos):.2%}; {np.mean(areas_pedacos < 0.01):.0%} abaixo de 1% da imagem")
print(f"- mais de um painel: {(stats['n_paineis'] > 1).mean():.0%} das imagens")
if len(tr):
    print(f"- cópias com rotação/espelho (entre as reconhecidas): {(confident['melhor_orientacao'] != 'identidade').mean():.0%}")
    print(f"- cópias com escala diferente (>10%): {np.mean(np.abs(np.log(tr['escala'])) > np.log(1.1)):.0%}")
med = sift.groupby("classe")["n_pares"].median()
print(f"- pares SIFT repetidos (mediana): autênticas {med.get('authentic', 0):.0f}, forjadas {med.get('forged', 0):.0f}")
print(f"- forjadas sem nenhum par SIFT: {(sift.loc[sift['classe'] == 'forged', 'n_pares'] == 0).mean():.0%}")

## 8. Discussão: o que torna o problema difícil

Os números da seção 7 confirmam ou derrubam cada ponto abaixo. Ajuste o texto depois de rodar com os
dados reais.

**1. A fraude não deixa rastro na região de origem.** Na *copy-move*, a origem é um pedaço intacto da
imagem original. Só a comparação com o destino a revela (seção 6.4). Isso descarta detectores que
procuram anomalias locais de ruído ou compressão como solução completa: eles podem achar o destino,
mas não a origem, e a máscara verdadeira contém as duas. O problema é de **correspondência dentro da
imagem**, não de classificação de textura. Isso favorece arquiteturas que comparam regiões entre si
(autocorrelação de atributos, casamento de pontos-chave).

**2. Semelhança não é fraude.** Figuras biomédicas são repetitivas por natureza: células do mesmo tipo,
bandas de *western blot* com o mesmo formato, painéis de controle parecidos, eixos e texto de
gráficos (seções 6.1 e 6.5). Se as autênticas têm uma quantidade de pares SIFT parecida com a das
forjadas, a autocorrespondência sozinha não separa as classes. A diferença está em **quão exata** é a
repetição: uma cópia é idêntica pixel a pixel (correlação perto de 1 na seção 6.3), duas células
diferentes apenas parecidas.

**3. A métrica é assimétrica e pune falsos positivos.** Prever qualquer máscara numa autêntica zera a
imagem, e as autênticas são uma parte grande do conjunto. É por isso que prever `authentic` para tudo
já pontua bem (0,304 no ranking privado). O pipeline precisa de uma **decisão de imagem conservadora**,
com limiar escolhido na validação, e só depois a segmentação.

**4. Cópias pequenas em imagens grandes.** A maioria dos pedaços cobre uma fração pequena da imagem
(seção 4), e ao reduzir a imagem para a entrada da rede muitas ficam com poucos pixels (seção 6.2).
Isso pede perdas que não sejam dominadas pelo fundo (Dice, focal), entrada em resolução maior ou
processamento em blocos, e um limiar de área mínima no pós-processamento escolhido com cuidado para
não apagar fraudes verdadeiras.

**5. Transformações geométricas.** Se parte das cópias foi espelhada, rotacionada ou reescalada
(seção 6.3), um casamento por correlação direta de blocos falha nelas. Descritores invariantes (SIFT,
atributos de redes treinadas com aumento de dados geométrico) e aumento de dados com espelhos e
rotações de 90° no treino tratam esses casos.

**6. Agrupar os pedaços em instâncias.** Cada máscara verdadeira é um par de cópias: origem e destino
juntos, em geral como dois pedaços separados (seção 3). A métrica casa máscaras uma a uma e divide pelo
maior número entre previstas e verdadeiras. Então separar a predição em componentes conexos e devolver
cada um como máscara dobra o número de instâncias e corta o escore pela metade, mesmo com os pixels
certos. Juntar duas cópias independentes numa máscara só também perde pontos. O pós-processamento
precisa **parear** os pedaços (por semelhança entre eles, como faz um casamento de pontos-chave) e
devolver cada par como uma instância.

**7. Heterogeneidade e tamanho.** Tamanhos de imagem muito variados, mistura de tons de cinza e cor,
modalidades diferentes (microscopia, *blots*, fotos, gráficos). Com poucos milhares de imagens, isso
favorece modelos pré-treinados (ImageNet, DINOv2) e aumento de dados, inclusive **fraudes sintéticas**
geradas copiando trechos das autênticas do treino, já com a máscara no formato de par.

**8. Vazamento.** Se um mesmo `case_id` aparece como autêntica e forjada (seção 1), as duas versões
precisam ficar na mesma partição, senão o teste mede memorização.